# Data Cleaning -- Test Dataset
**This notebook prepares Flatiron Health CSV files for patients with mestastatic colorectal cancer in anticipation of training a gradient-boosted survival model to predict mortality from time of first-line treatment. Specifically, it processes and cleans the test cohort.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorColorectal, merge_dataframes

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## Clean CSV files 

In [3]:
test_ids = pd.read_csv('../outputs/test_patient_ids.csv')

In [4]:
test_ids = test_ids.PatientID.to_list()

In [5]:
index_date_df = pd.read_csv('../data/LineOfTherapy.csv')

In [6]:
index_date_df = (
    index_date_df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [7]:
row_ID(index_date_df)

(38264, 37425)

In [8]:
index_date_df['StartDate'] = pd.to_datetime(index_date_df['StartDate'])

In [9]:
# Remove duplicate PatientIDs and select first for each patient based on StartDate
index_date_df = (
    index_date_df
    .sort_values(['PatientID', 'StartDate'], ascending = True)
    .drop_duplicates(subset='PatientID', keep='first')
)

In [10]:
row_ID(index_date_df)

(37425, 37425)

In [11]:
index_date_df = (
    index_date_df[index_date_df.PatientID.isin(test_ids)]
    [['PatientID', 'StartDate']]
)

In [12]:
row_ID(index_date_df)

(7485, 7485)

In [13]:
processor = DataProcessorColorectal()

### Process Enhanced_MetastaticCRC.csv

In [14]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticCRC.csv',
                                         patient_ids = test_ids,
                                         drop_stage = True,
                                         drop_dates = False)

2026-03-21 20:07:55,253 - INFO - Successfully read Enhanced_MetastaticCRC.csv file with shape: (46877, 5) and unique PatientIDs: 46877
2026-03-21 20:07:55,254 - INFO - Filtering for 7485 specific PatientIDs
2026-03-21 20:07:55,259 - INFO - Successfully filtered Enhanced_MetastaticCRC.csv file with shape: (7485, 5) and unique PatientIDs: 7485
2026-03-21 20:07:55,267 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (7485, 7) and unique PatientIDs: 7485


In [15]:
enhanced_df.shape

(7485, 7)

In [16]:
enhanced_df = pd.merge(enhanced_df, index_date_df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')

In [17]:
enhanced_df.shape

(7485, 8)

In [18]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])

In [19]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [20]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 based on training imputation
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [21]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [22]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2026-03-21 20:07:55,477 - INFO - Successfully read Demographics.csv file with shape: (46877, 6) and unique PatientIDs: 46877
2026-03-21 20:07:55,501 - INFO - Successfully processed Demographics.csv file with final shape: (7485, 6) and unique PatientIDs: 7485


In [23]:
demographics_df = demographics_df.copy()

In [24]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [25]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [26]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetCRCBiomarkers.csv

In [27]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-21 20:07:55,746 - INFO - Successfully read Enhanced_MetCRCBiomarkers.csv file with shape: (207027, 15) and unique PatientIDs: 42112
2026-03-21 20:07:55,806 - INFO - Successfully merged Enhanced_MetCRCBiomarkers.csv df with index_date_df resulting in shape: (36507, 16) and unique PatientIDs: 7026
2026-03-21 20:07:55,941 - INFO - Successfully processed Enhanced_MetCRCBiomarkers.csv file with final shape: (7485, 5) and unique PatientIDs: 7485


In [28]:
biomarkers_df['BRAF_status'] = biomarkers_df['BRAF_status'].fillna('unknown')
biomarkers_df['KRAS_status'] = biomarkers_df['KRAS_status'].fillna('unknown')
biomarkers_df['NRAS_status'] = biomarkers_df['NRAS_status'].fillna('unknown')
biomarkers_df['MMR/MSI_status'] = biomarkers_df['MMR/MSI_status'].fillna('unknown')

### Process Enhanced_CRC_HER2.csv

In [29]:
her2_df = processor.process_her2(file_path = '../data/Enhanced_CRC_HER2.csv',
                                 index_date_df = index_date_df, 
                                 index_date_column = 'StartDate',
                                 days_before = None, 
                                 days_after = 30)

2026-03-21 20:07:55,978 - INFO - Successfully read Enhanced_CRC_HER2.csv file with shape: (31958, 8) and unique PatientIDs: 18810
2026-03-21 20:07:55,993 - INFO - Successfully merged Enhanced_CRC_HER2.csv df with index_date_df resulting in shape: (5769, 9) and unique PatientIDs: 3319
2026-03-21 20:07:56,029 - INFO - Successfully processed Enhanced_CRC_HER2.csv file with final shape: (7485, 3) and unique PatientIDs: 7485


In [30]:
her2_df['HER2_status'] = her2_df['HER2_status'].fillna('unknown')

In [31]:
her2_df['HER2_percent_staining'] = her2_df['HER2_percent_staining'].cat.add_categories(['unknown'])
her2_df['HER2_percent_staining'] = her2_df['HER2_percent_staining'].fillna('unknown')

### Process ECOG.csv

In [32]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-21 20:07:56,430 - INFO - Successfully read ECOG.csv file with shape: (1017535, 4) and unique PatientIDs: 36246
2026-03-21 20:07:56,609 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (195016, 5) and unique PatientIDs: 6180
2026-03-21 20:07:56,705 - INFO - Successfully processed ECOG.csv file with final shape: (7485, 3) and unique PatientIDs: 7485


In [33]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [34]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [35]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-21 20:08:16,786 - INFO - Successfully read Vitals.csv file with shape: (17264775, 16) and unique PatientIDs: 46724
2026-03-21 20:08:23,669 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (3136607, 17) and unique PatientIDs: 7483
2026-03-21 20:08:24,803 - INFO - Successfully processed Vitals.csv file with final shape: (7485, 8) and unique PatientIDs: 7485


### Process Lab.csv

In [36]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'cea': ['2039-6'], 'ldh': ['2532-0', '14804-9']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-21 20:09:42,161 - INFO - Successfully read Lab.csv file with shape: (47135286, 17) and unique PatientIDs: 44960
2026-03-21 20:10:00,254 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (8718442, 18) and unique PatientIDs: 7327
2026-03-21 20:10:18,208 - INFO - Successfully processed Lab.csv file with final shape: (7485, 86) and unique PatientIDs: 7485


### Process MedicationAdministration.csv

In [37]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-21 20:10:26,714 - INFO - Successfully read MedicationAdministration.csv file with shape: (6618222, 11) and unique PatientIDs: 38657
2026-03-21 20:10:28,632 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (1234210, 12) and unique PatientIDs: 6987
2026-03-21 20:10:28,701 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (7485, 9) and unique PatientIDs: 7485


### Process Diagnosis.csv

In [38]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-21 20:10:31,006 - INFO - Successfully read Diagnosis.csv file with shape: (3378971, 6) and unique PatientIDs: 46877
2026-03-21 20:10:31,589 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (572269, 7) and unique PatientIDs: 7485
2026-03-21 20:10:32,703 - INFO - Successfully processed Diagnosis.csv file with final shape: (7485, 40) and unique PatientIDs: 7485


### Process Enhanced_Mortality_V2.csv

In [39]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                           her2_path = '../data/Enhanced_CRC_HER2.csv',
                                           oral_path = '../data/Enhanced_MetCRC_Orals.csv', 
                                           progression_path = '../data/Enhanced_MetCRC_Progression.csv',
                                           drop_dates = False)

2026-03-21 20:10:32,733 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (31065, 2) and unique PatientIDs: 31065
2026-03-21 20:10:32,753 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (7485, 3) and unique PatientIDs: 7485
2026-03-21 20:10:34,968 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_her2_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-21 20:10:34,974 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (7485, 6) and unique PatientIDs: 7485. There are 0 out of 7485 patients with missing duration values


In [40]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [41]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [42]:
testing_df = merge_dataframes(enhanced_df,
                              demographics_df,
                              biomarkers_df,
                              her2_df,
                              ecog_df,
                              vitals_df,
                              labs_df,
                              medications_df,
                              diagnosis_df,
                              mortality_df,
                              merge_type = 'inner')

2026-03-21 20:10:35,045 - INFO - Anticipated number of merges: 9
2026-03-21 20:10:35,046 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 162
2026-03-21 20:10:35,047 - INFO - Dataset 1 shape: (7485, 7), unique PatientIDs: 7485
2026-03-21 20:10:35,048 - INFO - Dataset 2 shape: (7485, 6), unique PatientIDs: 7485
2026-03-21 20:10:35,050 - INFO - Dataset 3 shape: (7485, 5), unique PatientIDs: 7485
2026-03-21 20:10:35,051 - INFO - Dataset 4 shape: (7485, 3), unique PatientIDs: 7485
2026-03-21 20:10:35,052 - INFO - Dataset 5 shape: (7485, 4), unique PatientIDs: 7485
2026-03-21 20:10:35,053 - INFO - Dataset 6 shape: (7485, 8), unique PatientIDs: 7485
2026-03-21 20:10:35,054 - INFO - Dataset 7 shape: (7485, 86), unique PatientIDs: 7485
2026-03-21 20:10:35,055 - INFO - Dataset 8 shape: (7485, 9), unique PatientIDs: 7485
2026-03-21 20:10:35,056 - INFO - Dataset 9 shape: (7485, 40), unique PatientIDs: 7485
2026-03-21 20:10:35,056 - I

In [43]:
testing_df.shape

(7474, 162)

## Export dataframe

In [44]:
testing_df.to_csv('../outputs/1L_features_testing.csv', index = False)

In [45]:
# Save dtypes
testing_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_testing_dtypes.csv')